# SPEC figure code — getting started

This notebook checks that the environment and the data are in place, then builds
one panel end to end. Run it top to bottom; it takes well under a minute.

If anything here fails, `01_all_figures.ipynb` will fail the same way, so start
here.

**Prerequisites**

1. `pip install -e .` from the repository root (installs `spec_analytics`).
2. The deposited search outputs unpacked into the repository, so that
   `figure2/input/` sits beside `figure2/scripts/`. Nothing is ever written into
   the input tree.


## Paths

Set these two if the data or the figures live outside the repository. Both are
read once, when `spec_config` is imported in the next cell, so **run this cell
first**; if you change it later, restart the kernel.


In [ ]:
import os

# Where the deposited search outputs are, and where figures are written.
#
#   DATA_ROOT   holds figure1/input, figure2/input, ... exactly as deposited
#   OUTPUT_ROOT receives every PDF, PNG and _sourcedata.csv, plus the caches
#
# Leave both as None to use the repository itself: unpack the deposited archive
# so that figure2/input/ sits beside figure2/scripts/, and figures land in
# <repository>/output/, which is git-ignored. Nothing is ever written into the
# input tree either way.
#
# To keep them elsewhere, give absolute paths, for example
#   DATA_ROOT   = r'D:\SPEC_data'
#   OUTPUT_ROOT = r'D:\SPEC_figures'

DATA_ROOT = None      # None -> leave as the environment has it
OUTPUT_ROOT = None    # None -> leave as the environment has it

for _name, _value in (('SPEC_DATA_ROOT', DATA_ROOT),
                      ('SPEC_OUTPUT_ROOT', OUTPUT_ROOT)):
    if _value is not None:
        os.environ[_name] = str(_value)
    print(f'{_name:18s} {os.environ.get(_name, "(unset: repository default)")}')


In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt

# The repository root is the nearest parent holding spec_config.py, so the
# notebook works whether it is opened from notebooks/ or from the root.
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'spec_config.py').exists())
sys.path.insert(0, str(REPO))
import spec_config as cfg


def show(path, width=8.0):
    """Draw a saved panel PNG inline at its own aspect ratio."""
    img = mpimg.imread(path)
    h, w = img.shape[:2]
    fig, ax = plt.subplots(figsize=(width, width * h / w))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(Path(path).name, fontsize=9)
    plt.show()
    # Without this the panel appears twice: plt.show() renders it, and the
    # inline backend renders every still-open figure again at the end of the
    # cell.
    plt.close(fig)


def run_panel(figure, script, display=True, width=8.0):
    """Run one panel script in a fresh interpreter and show what it wrote.

    A separate process, not %run: each script is written to be run standalone,
    and this keeps one script's globals out of the next one's.
    """
    path = REPO / figure / 'scripts' / script
    if not path.exists():
        raise FileNotFoundError(path)
    out_dir = Path(cfg.OUTPUT_ROOT) / figure
    started = time.time()
    # UTF-8 on both sides, explicitly. On Windows a piped child defaults to the
    # ANSI code page, so a panel script printing 'log10' as a subscript would
    # otherwise die on the write, or arrive undecodable here.
    env = dict(os.environ, PYTHONIOENCODING='utf-8')
    proc = subprocess.run([sys.executable, path.name], cwd=path.parent, env=env,
                          capture_output=True, encoding='utf-8',
                          errors='replace')
    print(proc.stdout, end='')
    if proc.returncode != 0:
        print(proc.stderr[-3000:], file=sys.stderr)
        raise RuntimeError(f'{figure}/{script} exited {proc.returncode}')
    fresh = sorted(p for p in out_dir.glob('*.png')
                   if p.stat().st_mtime >= started - 1)
    print(f'\n[{figure}/{script}] {time.time() - started:.0f}s, '
          f'{len(fresh)} panel(s) written to {out_dir}')
    if display:
        for p in fresh:
            show(p, width=width)
    return fresh


def run_figure(figure, scripts, width=8.0):
    """Run every script of one figure, in the given order."""
    written = []
    for script in scripts:
        written += run_panel(figure, script, width=width)
    return written


## Where the code reads and writes

These are what `spec_config.py` resolved from the cell above.


In [ ]:
print('repository :', REPO)
print('data root  :', cfg.DATA_ROOT)
print('output root:', cfg.OUTPUT_ROOT)
print()
print('example input path:', cfg.cross_input('figure2'))


## Is the library installed?

In [ ]:
import spec_analytics as core

print('spec_analytics:', core.__file__)
print(len([n for n in dir(core) if n.startswith('plot_')]), 'plot functions')
print(len(core.PALETTE_SINGLE), 'colours in PALETTE_SINGLE')


## Is the data there?

One row per figure. A figure with no `input/` folder cannot be rebuilt — download
the deposited archive and unpack it into the repository root.

`supplementary_figure3` and `supplementary_figure4` legitimately have none: both read `figure2/input/`.


In [ ]:
import pandas as pd

FIGURES = [f'figure{i}' for i in range(1, 7)] + \
          [f'supplementary_figure{i}' for i in range(1, 8)]

rows = []
for figure in FIGURES:
    d = Path(cfg.DATA_ROOT) / figure / 'input'
    files = [p for p in d.rglob('*') if p.is_file()] if d.is_dir() else []
    rows.append({'figure': figure, 'input present': d.is_dir(),
                 'files': len(files),
                 'MB': round(sum(p.stat().st_size for p in files) / 1e6, 1)})
pd.DataFrame(rows)


## Build one panel

Figure 6c: distinct glycan compositions per glycoprotein, SAX SPEC against ISD.
Chosen because it is the quickest panel in the paper — a couple of seconds — so
it is a fast check that the whole chain works.

The script prints its own summary numbers, writes a PDF, a 300 dpi PNG and a
source-data CSV, and the cell shows the PNG.


In [ ]:
_ = run_panel('figure6', 'panel_c_glyco_depth.py')

## The source data

Every panel writes the exact values it plotted next to the figure, so any number
in the manuscript can be traced without re-running anything.


In [ ]:
csv = Path(cfg.OUTPUT_ROOT) / 'figure6' / 'panel_c_glyco_depth_sourcedata.csv'
pd.read_csv(csv).head()


## Next

`01_all_figures.ipynb` rebuilds every panel in the paper, one cell per figure.
Allow roughly 15 minutes for the whole notebook; individual figures are quick
apart from figure 5, which quantifies 164 single fibers.
